# Portfolio Optimization

This notebook uses historical returns and predicted volatility to build and compare portfolio strategies.

The goal is to compare simple portfolio baselines against optimized portfolios. The strategies include an equal-weight portfolio, a historical minimum-volatility portfolio, a predictive minimum-volatility portfolio, and a maximum Sharpe ratio portfolio.

# Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from scipy.optimize import minimize

import matplotlib.pyplot as plt

# Paths

In [2]:
INTEGRATED_DATA_PATH = Path("../../data/processed/integrated")
MODELING_PATH = Path("../../data/processed/modeling")
PORTFOLIO_OUTPUT_PATH = Path("../../data/processed/portfolio_optimization")

In [4]:
if not PORTFOLIO_OUTPUT_PATH.exists():
    PORTFOLIO_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"Directory {PORTFOLIO_OUTPUT_PATH} already exists.") 

Directory ..\..\data\processed\portfolio_optimization already exists.


# Load Data

## Daily Market Data

Using daily market data for returns

In [5]:
daily_market_data = pd.read_csv(
    INTEGRATED_DATA_PATH / "daily_market_data.csv",
    parse_dates=["Date"]
)

## Random Forest

Using this one because it performed the best in the comparison file

In [6]:
rf_predictions = pd.read_csv(
    MODELING_PATH / "random_forest" / "test_predictions.csv",
    parse_dates=["Date"]
)

### Validate Loading Data Worked

In [8]:
daily_market_data.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type
0,2018-01-02,AAPL,40.267082,NaN,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock
1,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
2,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
3,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock
4,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock


In [10]:
daily_market_data.shape

(42210, 11)

In [9]:
rf_predictions.head()

,Date,ticker,future_volatility_20d,predicted_future_volatility_20d
0,2024-01-02,MSFT,0.010424,0.013266
1,2024-01-02,PG,0.011748,0.011599
2,2024-01-02,CAT,0.015127,0.013530
3,2024-01-02,AGG,0.003369,0.004461
4,2024-01-02,KO,0.006132,0.008131


In [11]:
rf_predictions.shape

(10101, 4)

# Create Return Matrix

In [12]:
returns_matrix = daily_market_data.pivot(
    index="Date",
    columns="ticker",
    values="daily_return"
).sort_index()

Drop rows with missing vals

In [13]:
returns_matrix = returns_matrix.dropna()

### Check Values

In [14]:
returns_matrix.head()

ticker,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,...,PG,QQQ,SPY,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-03,-0.000174,0.000092,0.012775,0.001528,-0.002637,0.001019,-0.002196,0.005432,0.008381,0.004654,...,-0.001213,0.009717,0.006325,0.004781,0.010490,0.009955,-0.002903,0.006971,-0.020549,0.019640
2018-01-04,0.004645,-0.000641,0.004476,0.013734,0.005127,0.014325,0.014084,0.004463,0.017154,0.008801,...,0.007068,0.001750,0.004215,-0.000159,0.004340,0.003718,-0.017227,0.008307,0.003243,0.001384
2018-01-05,0.011385,-0.000641,0.016163,0.015805,-0.001036,-0.006419,-0.000217,0.012278,0.009060,0.012398,...,0.000658,0.010043,0.006664,-0.002855,0.019069,0.023949,0.000494,0.006694,-0.002281,-0.000806
2018-01-08,-0.003714,-0.000276,0.014425,0.025129,-0.000160,0.001477,-0.001519,-0.005083,-0.004611,0.001020,...,0.005261,0.003891,0.001829,-0.000637,-0.017357,0.004038,0.005182,-0.000512,-0.001715,0.004496
2018-01-09,-0.000114,-0.002752,0.004676,0.002409,-0.004628,0.005069,0.004999,-0.000813,0.007162,-0.000680,...,-0.007305,0.000061,0.002264,-0.013373,0.004983,-0.001927,-0.012888,0.000853,-0.003668,-0.004246


In [15]:
returns_matrix.shape

(2009, 21)

# Train/Test Split

In [16]:
train_returns = returns_matrix[returns_matrix.index < "2024-01-01"]
test_returns = returns_matrix[returns_matrix.index >= "2024-01-01"]

# Create Portfolio Helper Functions

In [17]:
TRADING_DAYS = 252

In [18]:
def portfolio_return(weights, mean_returns):
    return np.dot(weights, mean_returns) * TRADING_DAYS

In [19]:
def portfolio_volatility(weights, covariance_matrix):
    return np.sqrt(np.dot(weights.T, np.dot(covariance_matrix, weights))) * np.sqrt(TRADING_DAYS)

In [20]:
def portfolio_sharpe(weights, mean_returns, covariance_matrix, risk_free_rate=0.0):
    port_return = portfolio_return(weights, mean_returns)
    port_vol = portfolio_volatility(weights, covariance_matrix)

    if port_vol == 0:
        return 0

    return (port_return - risk_free_rate) / port_vol

# Create Portfolio Evaluation Functions

In [21]:
def max_drawdown(portfolio_returns):
    cumulative = (1 + portfolio_returns).cumprod()
    running_max = cumulative.cummax()
    drawdown = (cumulative - running_max) / running_max

    return drawdown.min()

In [22]:
def evaluate_portfolio(strategy_name, weights, returns_data, risk_free_rate=0.0):
    daily_portfolio_returns = returns_data.dot(weights)

    annual_return = daily_portfolio_returns.mean() * TRADING_DAYS
    annual_volatility = daily_portfolio_returns.std() * np.sqrt(TRADING_DAYS)

    if annual_volatility == 0:
        sharpe_ratio = 0
    else:
        sharpe_ratio = (annual_return - risk_free_rate) / annual_volatility

    cumulative_return = (1 + daily_portfolio_returns).prod() - 1
    drawdown = max_drawdown(daily_portfolio_returns)

    return {
        "strategy": strategy_name,
        "annualized_return": annual_return,
        "annualized_volatility": annual_volatility,
        "sharpe_ratio": sharpe_ratio,
        "max_drawdown": drawdown,
        "cumulative_return": cumulative_return
    }